<div align="center">
<img src="https://poorit.in/image.png" alt="Poorit" width="40" style="vertical-align: middle;"> <b>LPU — BACKEND & GENERATIVE AI</b>

## Day 2 (Bonus): Building UIs with Gradio — Interface & ChatInterface

**Lovely Professional University**
*Backend & Generative AI · Poorit Technologies*

</div>

---

### What You'll Learn

In this notebook, you will:

1. Turn any Python function into a web UI with **`gr.Interface`**
2. Back an interface with an **LLM** — and **stream** the reply
3. Build a chatbot in a few lines with **`gr.ChatInterface`**
4. Launch and share your demo straight from Colab

---

## 1. Environment Setup

Run these first. You'll need an **OpenAI API key**.

In [ ]:
# Install the packages we need
!pip install -q openai gradio

In [ ]:
# Imports
import os
from getpass import getpass
from openai import OpenAI
import gradio as gr

In [ ]:
# API key (typed securely - not shown on screen)
openai_api_key = getpass("Enter your OpenAI API Key: ")
os.environ["OPENAI_API_KEY"] = openai_api_key

openai_client = OpenAI()   # reads OPENAI_API_KEY from the environment
MODEL = "gpt-4o-mini"
print("Ready. Model:", MODEL)

## 2. Your First Interface

`gr.Interface` wraps **any function** — an input goes in, an output comes out — into a web UI. No HTML, no CSS.

In [ ]:
# Any normal Python function works.
def shout(text):
    return text.upper()

shout("hello gradio")

In [ ]:
# fn = your function · inputs/outputs = the widgets · .launch() starts the UI.
# In Colab it renders right here; add share=True inside launch() for a public link.
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch()

## 3. Back it with an LLM

Swap `shout` for a function that calls a model — now the textbox is an AI app.

In [ ]:
system_message = "You are a helpful assistant. Answer in markdown."

def ask_llm(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt},
    ]
    r = openai_client.chat.completions.create(model=MODEL, messages=messages)
    return r.choices[0].message.content

In [ ]:
# A more polished interface: a labelled box in, markdown out, plus example prompts.
gr.Interface(
    fn=ask_llm,
    inputs=gr.Textbox(label="Your question", lines=4),
    outputs=gr.Markdown(label="Answer"),
    title="Ask the AI",
    examples=["Explain APIs in 2 lines", "What is the capital of India?"],
    flagging_mode="never",
).launch()

## 4. Stream the reply

Waiting for the whole answer feels slow. **Stream** it: pass `stream=True`, then **`yield`** the text-so-far. Gradio shows it typing out — just like ChatGPT.

In [ ]:
def stream_llm(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt},
    ]
    stream = openai_client.chat.completions.create(model=MODEL, messages=messages, stream=True)

    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""   # or "" guards empty chunks
        yield result                                       # each yield updates the UI

In [ ]:
# Same Interface - just point it at the generator. Gradio streams whatever it yields.
gr.Interface(
    fn=stream_llm,
    inputs=gr.Textbox(label="Your question", lines=4),
    outputs=gr.Markdown(label="Answer"),
    title="Ask the AI (streaming)",
    flagging_mode="never",
).launch()

## 5. A chatbot with `gr.ChatInterface`

`gr.ChatInterface` gives you a **whole chat UI** — message box, chat bubbles, history — from one function `fn(message, history)`. Gradio stores the conversation for you.

In [ ]:
# history arrives as a list of {"role", "content"} dicts - the same shape as the API.
def chat(message, history):
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    r = openai_client.chat.completions.create(model=MODEL, messages=messages)
    return r.choices[0].message.content

gr.ChatInterface(fn=chat).launch()

Make the chatbot **stream** too — the same trick, `yield` the growing reply:

In [ ]:
def chat_stream(message, history):
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    stream = openai_client.chat.completions.create(model=MODEL, messages=messages, stream=True)

    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

gr.ChatInterface(fn=chat_stream).launch()

## 6. Exercises

Fill in the blanks (`___`) and run each cell.

### Q1: A word-counter Interface

Wrap this function in a `gr.Interface` (a textbox in, a textbox out).

**Hints:** `fn=` the function; `inputs="textbox"`.

In [ ]:
def word_count(text):
    return f"{len(text.split())} words"

gr.Interface(fn=___, inputs="___", outputs="textbox", flagging_mode="never").launch()

### Q2: A pirate chatbot

Build a `ChatInterface` whose system prompt makes it answer like a pirate.

**Hints:** set the system prompt; `fn=` your function. Nothing else is needed — `history` already
arrives in the API's message format.

In [ ]:
pirate_system = "___"   # e.g. "You are a pirate. Answer everything in pirate slang."

def pirate_chat(message, history):
    messages = [{"role": "system", "content": ___}] + history + [{"role": "user", "content": message}]
    r = openai_client.chat.completions.create(model=MODEL, messages=messages)
    return r.choices[0].message.content

gr.ChatInterface(fn=___, type="___").launch()

---

### ✅ Key Takeaways

- **`gr.Interface(fn, inputs, outputs)`** turns any function into a web UI — no front-end code.
- Point `fn` at an LLM and you have an AI app in a few lines.
- **Stream** by making `fn` a generator that **`yield`s** the growing text.
- **`gr.ChatInterface(fn)`** builds a full chatbot; `fn(message, history)` gets the history for free, already as `{"role", "content"}` dicts.
- `.launch(share=True)` gives a public link you can send to anyone.